<a href="https://colab.research.google.com/github/rishikreddyv/gemini-rag-agent-project/blob/main/gemini_rag_agent_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q langchain langchain-community langchain-google-genai faiss-cpu pypdf sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 21.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY_HERE"

In [ ]:
from google.colab import files
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

Saving DeltaX.pdf to DeltaX.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(pdf_path)
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(f"{len(chunks)} chunks created")

5 chunks created


In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/tmp/ipykernel_4481/316697040.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (4 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

In [ ]:
import os
key = os.environ.get("GOOGLE_API_KEY", "NOT SET")
print(f"Key starts with: {key[:8]}..." if key != "NOT SET" else "NOT SET")

Key starts with: AQ.Ab8RN...


In [ ]:
query = "Summarize this document in one sentence"
relevant_docs = retriever.invoke(query)
context = "\n\n".join([doc.page_content for doc in relevant_docs])
prompt = f"""Answer the question based only on the context below.
Context:
{context}
Question: {query}
Answer:"""
response = llm.invoke(prompt)
print(response.content)

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


[{'type': 'text', 'text': 'This document presents the resume of Vendidandi Rishik Reddy, a Computer Science undergraduate at K L University specializing in AI-Driven Language Technologies, highlighting his skills and projects in full-stack web development and natural language processing.', 'extras': {'signature': 'EqMPCqAPARFNMg+5jZvhDbWvXcRiEYfjnaLUAGZJzztPLGG3RdL4S34a/T7CLHMst7lj5D0hFi64p8w23dYKuIprfYWqWlnWXbibuG6oufk5jCzowCXVevzUGcJsuMYm2qVaiR4He2l/+02YroLP42hsFk/R+UMlaR9PptAcd90awK2YUBxNinsl5EOUstyI/H8gDoWVKhcxG+SEK6EXRF+bbZjN+HqVrCAF6LPASaa/+9qLuMDjKfCPkqSlTOsgilRf6K79sgOQ2AerKKwbu9+XOcIWU2OBeyZVRL/YulaFa9BrUO1QI5pb69Gn3vmer3krcHYBw9h1TFTrqQyb32MvLGdkJ+SQXdLBDw4hG4kdjRXlsM0aCNqPHG/ZcO4LJtbmvnxYfstgPQMJdgh6ljWg6dhW/1Rpoz0Tp8JZlXzEHr9QojiNDU8yAeNXZ0AnGIwuY4AYhpNE0N03V3fZRVqcl5scrtrlv+yYEnhvNjrp7ravjFjEj7faVqIQGHhK6NXFCLEVfpcDQftXPCB1wIJ3FlLat7SQgJyrYECjtRM+o3ylz/rlYPXoTxn/I6bRpx3ZhHRbVOIM/l3B5jop+mgbpuTilMO9FsmlAOhrkE+PeKk+7uzhLqr+oMk7osIQP+l+CUqur+pO8TvtJQitDXMAQUOg4KaebqNdtzyqK1pN

In [ ]:
import os
key = os.environ.get("GOOGLE_API_KEY", "NOT SET")
print(f"Length: {len(key)}")
print(f"First 6 chars: {key[:6]}")
print(f"Last 4 chars: {key[-4:]}")

Length: 53
First 6 chars: AQ.Ab8
Last 4 chars: QsjQ


In [ ]:
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="Say hello in one word"
)
print(response.text)

Hello


In [ ]:
!pip install -q -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 78.0 MB/s eta 0:00:00


In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.tools import tool

search = DuckDuckGoSearchRun()

@tool
def calculator(expression: str) -> str:
    """Use this for math. Input must be a plain math expression like '25*4'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def web_search(query: str) -> str:
    """Use this to answer questions about current events or facts you don't know."""
    return search.run(query)

tools = [calculator, web_search]

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

agent_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)
agent = create_react_agent(agent_llm, tools)

/tmp/ipykernel_4481/2245278577.py:5: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(agent_llm, tools)


In [ ]:
response = agent.invoke({"messages": [("user", "What is 47 * 23, and who is the current CEO of OpenAI?")]})

for message in response["messages"]:
    print(f"{message.type}: {message.content}")

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


human: What is 47 * 23, and who is the current CEO of OpenAI?
ai: []
tool: 1081
tool: 10 hours ago - Altman co-founded OpenAI in 2015 and became its CEO in 2019, a role that made him a prominent figure of the AI boom. He supervised the launch of ChatGPT in November 2022. In 2023, he was ousted by the organization's board of directors for not being "consistently candid". 12 hours ago - In June 2026 it filed for an initial public offering. OpenAI has faced multiple lawsuits for alleged deaths linked to its chatbots and copyright infringement against authors and media companies. In November 2023, OpenAI's board removed Sam Altman as CEO, citing a lack of confidence, but reinstated ... 1 week ago - Sam Altman says OpenAI has fumbled at communicating AI benefits • Sep. 4, 2026, 10:40 AM ET(The San Jose Mercury News)...(Show more) ... Sam Altman was president of the start-up accelerator Y Combinator from 2014 to 2019 and became CEO of ... July 22, 2026 - OpenAI CEO Sam Altman has become one 